# PROYEK AKHIR: EDA Starter Project

Nama Anggota Kelompok :
1. Amadeo Abqary Dhiyaulhaq
2. Muhammad Amzar Hamizan

Dataset Yang Dipilih : Dataset Penjualan Kantin/Toko
Pertanyaan Analisis Awal :
- Produk apa yang paling laris?
- kasir mana yang paling banyak transaksi?
- Metode pembayaran apa yang paling sering dipakai?

Dugaan masalah Kualitas data </br>
- Missing value pada kolom : jumlah_terjual, harga_satuan, nama_kasir
- Duplikat Data
- Tipe data tidak sesuai pada kolom : jumlah_terjual, harga_Satuan, Tanggal
- kategori tidak konsisten kapitalisasi Makanan/makanan/MAKANAN
- Tanggalnya tidak sesuai

Rencana Teknik Pembersihan :
- Hapus data duplikat
- Data kosong di isi dengan nilai rata-rata
- hapus tulisan "pcs" dan Rp agar jadi angka
- samakan penulisan kategori

In [32]:
import numpy as np
import pandas as pd

#Data Loading
df = pd.read_csv("dataset_penjualan_kantin.csv")

print(df.head())        # 5 Baris Pertama
print()

  id_transaksi     tanggal  nama_produk kategori jumlah_terjual harga_satuan  \
0      TRX0042  2026-08-11   Roti Bakar  Makanan              9         7000   
1      TRX0005  2026-08-03     Gorengan  makanan              2         2000   
2      TRX0011  2026-08-04  Jus Alpukat  Minuman              4          NaN   
3      TRX0035  2026-08-10     Mie Ayam  Makanan            NaN        10000   
4      TRX0007  2026-08-03      Kerupuk    Snack             10      Rp2.000   

  nama_kasir metode_pembayaran  
0   Pak Agus             Tunai  
1     Bu Sri              QRIS  
2    Bu Wati             Tunai  
3   Pak Joko          Transfer  
4    Bu Wati          Transfer  



# 1. Data Inpection dan Loading

In [33]:
# Data Inspection
print(df.info())        # tipe data dan jumlah non-null
print()

print(df.describe())    # statistik ringkas kolom numerik
print()

print("Jumlah baris dan kolom adalah : ", df.shape)         # jumlah (baris dan kolom)

<class 'pandas.DataFrame'>
RangeIndex: 69 entries, 0 to 68
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id_transaksi       69 non-null     str  
 1   tanggal            69 non-null     str  
 2   nama_produk        69 non-null     str  
 3   kategori           69 non-null     str  
 4   jumlah_terjual     65 non-null     str  
 5   harga_satuan       66 non-null     str  
 6   nama_kasir         66 non-null     str  
 7   metode_pembayaran  69 non-null     str  
dtypes: str(8)
memory usage: 4.4 KB
None

       id_transaksi     tanggal  nama_produk kategori jumlah_terjual  \
count            69          69           69       69             65   
unique           65          27           12        9             21   
top         TRX0042  2026-08-11  Jus Alpukat  Makanan              9   
freq              2           7           11       19             12   

       harga_satuan nama_kasir metode_pembayar

# 2. DATA CLEANING

In [34]:
# DATA CLEANING

# Cek Missing Value
print("Data Kosong :")
print(df.isnull().sum())
print()

# Cek Data Duplikat
print("Data Duplikat :")
print(df.duplicated().sum())
print()

# Cek Tipe Data
print("Tipe Data :")
print(df.dtypes)

Data Kosong :
id_transaksi         0
tanggal              0
nama_produk          0
kategori             0
jumlah_terjual       4
harga_satuan         3
nama_kasir           3
metode_pembayaran    0
dtype: int64

Data Duplikat :
4

Tipe Data :
id_transaksi         str
tanggal              str
nama_produk          str
kategori             str
jumlah_terjual       str
harga_satuan         str
nama_kasir           str
metode_pembayaran    str
dtype: object


In [35]:
# Memberihkan Kolom "harga_satuan"

#Mengubah prefix "Rp" dan pemiah ribuan ".", lalu mengubah ke numerik
df['harga_satuan'] = (df['harga_satuan'].astype(str).replace('Rp', '', regex=False))
df['harga_satuan'] = (df['harga_satuan'].astype(str).replace('.', '', regex=False))
df['harga_satuan'] = (df['harga_satuan'].astype(str).replace('nan', np.nan))

df['harga_satuan'] = pd.to_numeric(df['harga_satuan'], errors='coerce')
df['harga_satuan'].describe()

count       52.000000
mean      7423.076923
std       3339.285786
min       2000.000000
25%       4750.000000
50%       8000.000000
75%      10000.000000
max      12000.000000
Name: harga_satuan, dtype: float64

In [36]:
#Merbersihkan Kolom "jumlah_terjual"

#Menghapu akhiran "pcs", lalu mengubahnya ke numerik
df['jumlah_terjual'] = (df['jumlah_terjual'].astype(str).str.replace('pcs', '', regex=False).str.strip().replace('nan', np.nan))

df['jumlah_terjual'] = pd.to_numeric(df['jumlah_terjual'], errors='coerce')
df['jumlah_terjual'].describe()

count     65.000000
mean      15.000000
std       61.252296
min        1.000000
25%        4.000000
50%        8.000000
75%       11.000000
max      500.000000
Name: jumlah_terjual, dtype: float64

In [37]:
# Cek outlier:** nilai maksimum `jumlah_terjual` terlihat jauh lebih besar dari
# nilai lain (kemungkinan salah input, misalnya `500` yang seharusnya `50`).

df[df["jumlah_terjual"] > 100]


,id_transaksi,tanggal,nama_produk,kategori,jumlah_terjual,harga_satuan,nama_kasir,metode_pembayaran
59,TRX0050,2026-08-12,Roti Bakar,MAKANAN,500.0,7000.0,Pak Joko,QRIS


In [38]:
# Nilai 500 pcs untuk satu transaksi kantin sekolah tidak masuk akal (outlier/typo).
# Karena tidak bisa dipastikan angka yang benar, baris ini diberi nilai kosong (NaN)
# supaya nanti ditangani bersama proses missing value, bukan dihapus langsung.
df.loc[df['jumlah_terjual'] > 100, 'jumlah_terjual'] = np.nan

In [39]:
#Membersihkan Kolom Tanggal
#Menyatukan tiga format tanggal (YYYY-MM-DD, DD/MM/YYYY, DD Bulan YYYY) Menjadi atu format datetime

bulan_map = {
    'Januari':'January',
    'Februari':'February',
    'Maret':'March',
    'April':'April',
    'Mei':'May',
    'Juni':'June',
    'Juli':'July',
    'Agustus':'August',
    'September':'September',
    'Oktober':'October',
    'November':'November',
    'Desember':'December'
}

def ke_bahasa_inggris(teks):
    for id_, en in bulan_map.items():
        teks = teks.replace(id_, en)
    return teks

def parse_tanggal(teks):
    teks = ke_bahasa_inggris(teks)
    for fmt in ('%Y-%m-%d', '%d/%m/%Y', '%d %B %Y'):
        try:
            return pd.to_datetime(teks, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df['tanggal'] = df['tanggal'].apply(parse_tanggal)
df['tanggal'].head(10)

0   2026-08-11
1   2026-08-03
2   2026-08-04
3   2026-08-10
4   2026-08-03
5   2026-08-11
6   2026-08-14
7   2026-08-04
8   2026-08-07
9   2026-08-13
Name: tanggal, dtype: datetime64[us]

In [40]:
#Menyamakan Kapital kolom "Kategori"

print("Sebelum:", df['kategori'].unique())
df['kategori'] = df['kategori'].str.strip().str.title()
print("Sesudah:", df['kategori'].unique())

Sebelum: <StringArray>
['Makanan', 'makanan', 'Minuman',   'Snack', 'MINUMAN',   'snack', 'MAKANAN',
 'minuman',   'SNACK']
Length: 9, dtype: str
Sesudah: <StringArray>
['Makanan', 'Minuman', 'Snack']
Length: 3, dtype: str


In [41]:
#Menangani Missing Value

df['harga_satuan'] = df.groupby('nama_produk')['harga_satuan'].transform(lambda x: x.fillna(x.median()))
df['jumlah_terjual'] = df.groupby('nama_produk')['jumlah_terjual'].transform(lambda x: x.fillna(x.median()))
df['nama_kasir'] = df['nama_kasir'].fillna('Tidak Diketahui')

df.isnull().sum()

id_transaksi         0
tanggal              0
nama_produk          0
kategori             0
jumlah_terjual       0
harga_satuan         0
nama_kasir           0
metode_pembayaran    0
dtype: int64

In [42]:
# Menghapus Duplikat dan Mematikan Tipe Data

jumlah_sebelum = len(df)
df = df.drop_duplicates()
jumlah_sesudah = len(df)
print(f"Baris sebelum: {jumlah_sebelum}, sesudah: {jumlah_sesudah}, terhapus: {jumlah_sebelum - jumlah_sesudah}")

Baris sebelum: 69, sesudah: 65, terhapus: 4


In [43]:
df['jumlah_terjual'] = df['jumlah_terjual'].astype(int)
df['harga_satuan'] = df['harga_satuan'].astype(int)
df.dtypes

id_transaksi                    str
tanggal              datetime64[us]
nama_produk                     str
kategori                        str
jumlah_terjual                int64
harga_satuan                  int64
nama_kasir                      str
metode_pembayaran               str
dtype: object

# 3. Data Manipulation

In [44]:
# Filtering (produk dengan penjualan laris (jumlah_terjual >10) )
laris = df[df['jumlah_terjual'] > 10]
laris.head()

,id_transaksi,tanggal,nama_produk,kategori,jumlah_terjual,harga_satuan,nama_kasir,metode_pembayaran
6,TRX0061,2026-08-14,Teh Botol,Minuman,14,5000,Bu Sri,QRIS
7,TRX0008,2026-08-04,Mie Ayam,Makanan,14,10000,Pak Joko,Tunai
9,TRX0057,2026-08-13,Jus Alpukat,Minuman,14,8000,Bu Sri,Tunai
12,TRX0047,2026-08-12,Teh Botol,Minuman,12,5000,Pak Joko,QRIS
14,TRX0048,2026-08-12,Keripik Singkong,Snack,15,3000,Bu Wati,QRIS


In [45]:
#Sorting (Transaksi dengan jumlah tertinggi)
df_urut = df.sort_values(by='jumlah_terjual', ascending=False)
df_urut.head(10)

,id_transaksi,tanggal,nama_produk,kategori,jumlah_terjual,harga_satuan,nama_kasir,metode_pembayaran
14,TRX0048,2026-08-12,Keripik Singkong,Snack,15,3000,Bu Wati,QRIS
20,TRX0046,2026-08-11,Nasi Goreng,Makanan,15,12000,Bu Wati,QRIS
16,TRX0054,2026-08-13,Jus Alpukat,Minuman,15,8000,Pak Joko,Tunai
9,TRX0057,2026-08-13,Jus Alpukat,Minuman,14,8000,Bu Sri,Tunai
7,TRX0008,2026-08-04,Mie Ayam,Makanan,14,10000,Pak Joko,Tunai
29,TRX0029,2026-08-07,Nasi Uduk,Makanan,14,8000,Bu Sri,QRIS
6,TRX0061,2026-08-14,Teh Botol,Minuman,14,5000,Bu Sri,QRIS
57,TRX0049,2026-08-12,Es Teh,Minuman,14,3000,Bu Sri,QRIS
30,TRX0051,2026-08-12,Mie Ayam,Makanan,13,10000,Bu Wati,Tunai
46,TRX0021,2026-08-06,Jus Alpukat,Minuman,13,8000,Pak Joko,QRIS


In [46]:
#Kolom Turunan (Pendapatan Per Transaksi)
df['total_pendapatan'] = df['harga_satuan'] * df['jumlah_terjual']
df[['nama_produk', 'harga_satuan', 'jumlah_terjual', 'total_pendapatan']].head()

,nama_produk,harga_satuan,jumlah_terjual,total_pendapatan
0,Roti Bakar,7000,9,63000
1,Gorengan,2000,2,4000
2,Jus Alpukat,8000,4,32000
3,Mie Ayam,10000,4,40000
4,Kerupuk,2000,10,20000


In [47]:
# Group by / Agregasi (Menjawab pertanyaan analisis awal)

# 1. Produk apa yang paling laris (berdasarkan total unit terjual)?
produk_laris = df.groupby('nama_produk')['jumlah_terjual'].sum().sort_values(ascending=False)
produk_laris

nama_produk
Nasi Goreng         83
Jus Alpukat         62
Teh Botol           58
Mie Ayam            55
Kerupuk             54
Keripik Singkong    36
Bakso               32
Es Teh              31
Roti Bakar          30
Nasi Uduk           21
Gorengan             9
Es Jeruk             9
Name: jumlah_terjual, dtype: int64

In [48]:
# 2. Kasir mana yang paling banyak transaksi?
transaksi_kasir = df['nama_kasir'].value_counts()
transaksi_kasir

nama_kasir
Pak Joko           20
Bu Wati            19
Bu Sri             13
Pak Agus           10
Tidak Diketahui     3
Name: count, dtype: int64

In [49]:
# 3. Metode pembayaran apa yang paling sering dipakai?
metode_favorit = df['metode_pembayaran'].value_counts()
metode_favorit

metode_pembayaran
QRIS        35
Tunai       23
Transfer     7
Name: count, dtype: int64

In [50]:
# Tambahan: total pendapatan per kategori produk
pendapatan_kategori = df.groupby('kategori')['total_pendapatan'].sum().sort_values(ascending=False)
pendapatan_kategori

kategori
Makanan    2294000
Minuman     915000
Snack       216000
Name: total_pendapatan, dtype: int64

## 4 Menyimpan Dataset Bersih

In [51]:
df.to_csv('dataset_penjualan_kantin_bersih.csv', index=False)
print("Dataset bersih tersimpan sebagai dataset_bersih.csv")
df.head()

Dataset bersih tersimpan sebagai dataset_bersih.csv


,id_transaksi,tanggal,nama_produk,kategori,jumlah_terjual,harga_satuan,nama_kasir,metode_pembayaran,total_pendapatan
0,TRX0042,2026-08-11,Roti Bakar,Makanan,9,7000,Pak Agus,Tunai,63000
1,TRX0005,2026-08-03,Gorengan,Makanan,2,2000,Bu Sri,QRIS,4000
2,TRX0011,2026-08-04,Jus Alpukat,Minuman,4,8000,Bu Wati,Tunai,32000
3,TRX0035,2026-08-10,Mie Ayam,Makanan,4,10000,Pak Joko,Transfer,40000
4,TRX0007,2026-08-03,Kerupuk,Snack,10,2000,Bu Wati,Transfer,20000
